#  EXP_002: Cálculo de π(x)

En este notebook extraeremos extraeremos los resultados de nuestro primer experimento (El cómputo de π(x)). Jugaremos con las distintas variables de entrada que nuestro códgos permiten como entrada.

In [2]:
import subprocess 

# CAMBIAR AL PATH LOCAL
BASE_DIR = "/home/usc/cursos/curso1285/home/TFG"

# Script
output = subprocess.run([f"{BASE_DIR}/problemas/Problema_002/scripts/prepare_002.sh", BASE_DIR], capture_output=True, text=True)

print(output.stdout)
# Ejecutables
P002_SEC   = f"{BASE_DIR}/problemas/Problema_002/execs/P_002_seq"
P002_OMP   = f"{BASE_DIR}/problemas/Problema_002/execs/P_002_omp"
P002_ESP_1 = f"{BASE_DIR}/problemas/Problema_002/execs/P_002_esp_1"
P002_ESP_2 = f"{BASE_DIR}/problemas/Problema_002/execs/P_002_esp_2"

# CSVs
R002_SEC_CSV   = f"{BASE_DIR}/problemas/Problema_002/results/P_002_sec.csv"
R002_OMP_CSV   = f"{BASE_DIR}/problemas/Problema_002/results/P_002_omp.csv"
R002_ESP_1_CSV = f"{BASE_DIR}/problemas/Problema_002/results/P_002_esp_1.csv"
R002_ESP_2_CSV = f"{BASE_DIR}/problemas/Problema_002/results/P_002_esp_2.csv"

# CONTROL BUCLES
M = 10    # Numero de repeticiones de cada configuracion
N_VALUES = [int(1e2),int(1e3), int(1e4), int(1e5), int(5e5), int(1e6), int(5e6), int(1e7)]   # Tamaño del problema
T_VALUES = [2, 4, 6, 8, 16]    # Numero de hilos
P_VALUES = [10]     
CHUNK_VALUES = [0,1,4,10,50]
SCHED_VALUES = ["static","dynamic"]
THRESHOLD_VALUES = [10, 25, 50, 100, 1000]


Primeramente, obtenemos los tiempos de ejecución de la versión secuencial, de esta versión únicamente obtenemos dos métricas.
- **Tiempo:** Duración del cómputo de PI.
- **Memoria usada:** Tamaño máximo del RSS.

In [4]:
import subprocess
import pandas as pd
import statistics
import csv

results_002_seq = []

# Computo del experimento.
for N in N_VALUES:
    times_002_seq = []
    rss_002_seq = []   
    for i in range(M):
        output = subprocess.run(
            [f"{BASE_DIR}/include/run.sh",P002_SEC,str(N)],
            capture_output=True,
            text=True
        )
        # Tiempo
        for line in output.stdout.splitlines(): 
            
            if line.startswith("TIME_TOT"):
                times_002_seq.append(float(line.split("=")[1].strip()))
                break
        
        # RSS
        for line in output.stdout.splitlines():
            if line.startswith("RSS = "):
                rss_002_seq.append(float(line.split("=")[1].strip()))
                break

    times_002_seq_mean = sum(times_002_seq) / M
    rss_002_seq_mean = sum(rss_002_seq) / M
    
    results_002_seq.append({
        "N": N,
        "TIME_mean": times_002_seq_mean,
        "RSS_mean": rss_002_seq_mean,
        "TIME_std": statistics.stdev(times_002_seq),
        "RSS_std": statistics.stdev(rss_002_seq),
    })

# Pasamos los resultados a un csv
with open(R002_SEC_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["N", "TIME_mean", "RSS_mean", "TIME_std", "RSS_std"])
    
    writer.writeheader()   
    
    for row in results_002_seq:
        writer.writerow(row)
print("SEC FINALIZADO")

KeyboardInterrupt: 

Una vez hemos obtenido los tiempos secuenciales, pasemos a la versión con puro OpenMP.

In [ ]:
import subprocess
import pandas as pd
import statistics
import csv
import os

results_002_omp = []

# Computo del experimento.
for N in N_VALUES:
    for T in T_VALUES:
        for sched in SCHED_VALUES:
            for chunk in CHUNK_VALUES:
                if chunk == 0:
                    os.environ["OMP_SCHEDULE"] = f"{sched}"
                else:
                    os.environ["OMP_SCHEDULE"] = f"{sched},{chunk}"
                    

                times_002_omp = []
                rss_002_omp = []   
                for i in range(M):
                    output = subprocess.run(
                        [f"{BASE_DIR}/include/run.sh",P002_OMP, str(N), str(T)],
                        capture_output=True,
                        text=True
                    )

                    # Tiempo
                    for line in output.stdout.splitlines(): 
                        
                        if line.startswith("TIME_TOT"):
                            times_002_omp.append(float(line.split("=")[1].strip()))
                            break
                    
                    # RSS
                    for line in output.stdout.splitlines():
                        if line.startswith("RSS"):
                            rss_002_omp.append(float(line.split("=")[1].strip()))
                            break

                    times_002_omp_mean = sum(times_002_omp) / M
                    rss_002_omp_mean = sum(rss_002_omp) / M
                    
                results_002_omp.append({
                    "N": N,
                    "T": T,
                    "sched": sched,
                    "chunk": chunk,
                    "TIME_mean": times_002_omp_mean,
                    "RSS_mean": rss_002_omp_mean,
                    "TIME_std": statistics.stdev(times_002_omp),
                    "RSS_std": statistics.stdev(rss_002_omp),
                })

    # Pasamos los resultados a un csv
    with open(R002_OMP_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["N", "T", "sched", "chunk", "TIME_mean", "RSS_mean", "TIME_std", "RSS_std"])
        
        writer.writeheader()   
        
        for row in results_002_omp:
            writer.writerow(row)
print("OMP FINALIZADO")

OMP FINALIZADO


Ahora las versiones especulativas. De las versiones especualtivas pretendemos extraer más métricas interesantes:
- **Tiempo decisión:** Cantidad de tiempo que el sistema invierte escogiendo la planificación más adecuada. Esto sería el Overhead que nuestra propia écnica introduce.
- **Winner:** La planificación que ha escogido como vencedora.

In [ ]:
import subprocess
import pandas as pd
import statistics
import csv

results_002_esp_1 = []

# Computo del experimento.
for N in N_VALUES:
    for T in T_VALUES:
        for p in P_VALUES:
            for chunk in CHUNK_VALUES:
                if T == 2:
                    continue
                
                if (N * p / (T*100)) < 1:   # Evitamos entrar en un bucle por tener demasiado hilos cuando chunk = 0
                    continue
                
                times_002_esp_1 = []
                times_dec_002_esp_1 = []
                rss_002_esp_1 = []   
                winners_002_esp_1 = []
                for i in range(M):
                    output = subprocess.run(
                        [f"{BASE_DIR}/include/run.sh",P002_ESP_1, str(N),str(T), str(p), str(chunk)],
                        capture_output=True,
                        text=True
                    )
                    # Tiempo
                    for line in output.stdout.splitlines(): 
                        if line.startswith("TIME_TOT"):
                            times_002_esp_1.append(float(line.split("=")[1].strip()))
                            break
                    
                    for line in output.stdout.splitlines(): 
                        if line.startswith("TIME_DEC"):
                            times_dec_002_esp_1.append(float(line.split("=")[1].strip()))
                            break
                    
                    for line in output.stdout.splitlines(): 
                        if line.startswith("WINNER"):
                            winners_002_esp_1.append(line.split("=")[1].strip())
                            break
                    
                    for line in output.stdout.splitlines(): 
                        if line.startswith("RSS"):
                            rss_002_esp_1.append(float(line.split("=")[1].strip()))
                            break
                    
                    times_002_esp_1_mean = sum(times_002_esp_1) / M
                    times_dec_002_esp_1_mean = sum(times_dec_002_esp_1) / M
                    rss_002_esp_1_mean = sum(rss_002_esp_1) / M
                
                results_002_esp_1.append({
                    "N": N,
                    "T": T,
                    "p": p,
                    "chunk": chunk,
                    "TIME_mean": times_002_esp_1_mean,
                    "TIME_dec_mean": times_dec_002_esp_1_mean,
                    "RSS_mean": rss_002_esp_1_mean,
                    "TIME_std": statistics.stdev(times_002_esp_1),
                    "RSS_std": statistics.stdev(rss_002_esp_1),
                    "TIME_dec_std": statistics.stdev(times_dec_002_esp_1),
                    "S_pred": winners_002_esp_1.count('S')
                })

                # Pasamos los resultados a un csv
                with open(R002_ESP_1_CSV, "w", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=["N", "T", "p","chunk", "TIME_mean", "RSS_mean", "TIME_dec_mean", "TIME_std", "RSS_std", "TIME_dec_std", "S_pred"])
                    writer.writeheader()   
                    
                    for row in results_002_esp_1:
                        writer.writerow(row)

print("ESP_1 FINALIZADO.")



ESP_1 FINALIZADO.


A continuación extraeremos los resultados de la versión especulativa 2, esta versión incluye dos nuevos outputs:

- **Desbalance0:** Es el coeficiente del tiempo de las iteraciones.

- **DEV_STD:** Valor de la desviación típica entre iteraciones.

In [ ]:
import subprocess
import pandas as pd
import statistics
import csv

results_002_esp_2 = []

# Computo del experimento.
for N in N_VALUES:
    for T in T_VALUES:
        for p in P_VALUES:
            for thr in THRESHOLD_VALUES:
                for chunk in CHUNK_VALUES:
                    if(chunk == 0):
                        continue
                    times_002_esp_2 = []
                    times_dec_002_esp_2 = []
                    rss_002_esp_2 = []
                    desvs = []
                    desbalanceos = []   
                    winners_002_esp_2 = []
                    for i in range(M):
                        output = subprocess.run(
                            [f"{BASE_DIR}/include/run.sh",P002_ESP_2, str(N), str(T), str(p), str(thr), str(chunk)],
                            capture_output=True,
                            text=True
                        )

                        # Tiempo
                        for line in output.stdout.splitlines(): 
                            if line.startswith("TIME_TOT"):
                                times_002_esp_2.append(float(line.split("=")[1].strip()))
                                break
                        
                        for line in output.stdout.splitlines(): 
                            if line.startswith("TIME_DEC"):
                                times_dec_002_esp_2.append(float(line.split("=")[1].strip()))
                                break
                        
                        for line in output.stdout.splitlines(): 
                            if line.startswith("WINNER"):
                                winners_002_esp_2.append(line.split("=")[1].strip())
                                break
                        
                        for line in output.stdout.splitlines(): 
                            if line.startswith("RSS"):
                                rss_002_esp_2.append(float(line.split("=")[1].strip()))
                                break
                        
                        for line in output.stdout.splitlines(): 
                            if line.startswith("STD_DEV"):
                                desvs.append(float(line.split("=")[1].strip()))
                                break
                        
                        for line in output.stdout.splitlines(): 
                            if line.startswith("DESBALANCEO"):
                                desbalanceos.append(float(line.split("=")[1].strip()))
                                break
                        
                    results_002_esp_2.append({
                        "N": N,
                        "T": T,
                        "p": p,
                        "umbral": thr,
                        "chunk": chunk,
                        "TIME_mean": statistics.mean(times_002_esp_2),
                        "TIME_dec_mean": statistics.mean(times_dec_002_esp_2),
                        "RSS_mean": statistics.mean(rss_002_esp_2),
                        "desv_mean": statistics.mean(desvs),
                        "desbalanceos_mean": statistics.mean(desbalanceos),
                        "TIME_std": statistics.stdev(times_002_esp_2),
                        "RSS_std": statistics.stdev(rss_002_esp_2),
                        "TIME_dec_std": statistics.stdev(times_dec_002_esp_2),
                        "desv_std": statistics.stdev(desvs),    # DESVIACION ESTANDAR DEL DATASET
                        "desbalanceos_std": statistics.stdev(desbalanceos),
                        "S_pred": winners_002_esp_2.count('S')
                    })

                    # Pasamos los resultados a un csv
                    with open(R002_ESP_2_CSV, "w", newline="") as f:
                        writer = csv.DictWriter(f, fieldnames=["N", "T", "p", "umbral", "chunk", "TIME_mean", "RSS_mean", "TIME_dec_mean", "desv_mean", "desbalanceos_mean", "TIME_std", "RSS_std", "TIME_dec_std","desv_std", "desbalanceos_std", "S_pred"])
                        writer.writeheader()   
                        
                        for row in results_002_esp_2:
                            writer.writerow(row)

print("ESP_2 FINALIZADO")



ESP_2 FINALIZADO
